### Imports

In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from google import genai
import requests
from minsearch import Index # minsearch is a simple in-memory search engine

In [2]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

for model in client.models.list():
    print(model.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

### Setup

In [4]:
def llm(instructions="", user_prompt="", model="gemini-2.5-flash"):
    if not GOOGLE_API_KEY:
        raise RuntimeError("Set GOOGLE_API_KEY in your environment")
    
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    
    # Combine messages for the API call, preserving role information
    combined_content = ""
    for msg in message_history:
        combined_content += f"[{msg['role'].upper()}]\n{msg['content']}\n\n"
    
    response = client.models.generate_content(
        model=model,
        contents=combined_content,
    )
    return response.text

In [6]:
print(llm(user_prompt="What is the capital of Kenya? "))

The capital of Kenya is **Nairobi**.


In [7]:
question = "I just discovered the course. Can I join now?"
answer = llm(user_prompt=question)
print(answer)

That's fantastic! Welcome aboard.

Whether you can join right now depends on the nature of the course. Could you tell me a little more about it?

Specifically, it would be helpful to know:

1.  **What is the name of the course?**
2.  **Where did you discover it (e.g., specific platform like Coursera, Udemy, edX, a university website, a private academy, etc.)?**
3.  **Did you see any mention of start dates, enrollment periods, or "apply now" buttons on the page?**

Generally, courses fall into a few categories:

*   **Self-Paced & Always Open:** You can usually enroll and start immediately.
*   **Cohort-Based with Specific Start Dates:** These courses have fixed start and end dates, and you might have to wait for the next cohort, or there might be an upcoming enrollment window.
*   **Application-Based:** Some courses (especially longer programs or certifications) require an application process, which can take time.
*   **Enrollment Deadlines:** Even for seemingly flexible courses, there

In [8]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [9]:
prompt = f"""
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [12]:
answer = llm(user_prompt=prompt)
print(f"===Prompt===\n {prompt}")
print(f"===Answer===\n {answer}")

===Prompt===
 
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. St

### The Course FAQ Dataset

In [13]:
# This returns a list of courses. 
# Each course has a path field that points to its FAQ data.
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 84},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [15]:
# Let's fetch all the FAQ documents from all courses:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

print(f"Type of documents: {type(documents)}")
print(f"Number of documents: {len(documents)}")

print("=== First document ===")
documents[0]


Type of documents: <class 'list'>
Number of documents: 1349
=== First document ===


{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In the RAG pipeline, this dataset is our knowledge base:

- We index all the documents (the search step)
- When a student asks a question, we search the index
- The search returns the most relevant FAQ entries
- We give those entries to the LLM as context
- The LLM generates an answer based on the context

The question and answer fields contain the text we'll search through. The course field lets us filter by course. 
For example, if a student asks about the data engineering course, we skip results from the ML course. 
The section field helps with ranking - knowing which part of the course a question belongs to is useful context.

### Indexing with minsearch

In [16]:
# We'll index the question, section, and answer fields as text (they'll be tokenized and ranked), and the course field as a keyword (for filtering).
# The index tokenizes text fields and treats keyword fields as exact strings.
# Text fields are the fields you search through. When you type a query, the search engine looks at these fields and tokenizes them. 
# It splits text into words, lowercases them, removes stop words, and ranks the results by relevance. 
# So question, section, and answer are text fields.
# Keyword fields are for exact matching. 

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [17]:
# Let's try a search with the question we used before
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    # We use boost_dict to give the question field more weight (2.0 instead of the default 1.0) and section less weight (0.5)
    # Query words appearing in the question field is a stronger signal than them appearing in the section name.
    boost_dict={"question": 2.0, "section": 0.5},
    # We used filter_dict to only return results from the LLM Zoomcamp course. Without this filter, we'd get results from all four courses.
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5 # get the top 5 results
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [18]:
# We get back 5 results from the LLM Zoomcamp FAQ. 
# The best match is the FAQ entry "I just discovered the course. Can I still join?" which is exactly what we need.
# These are the candidate documents we'll send to the LLM.
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [19]:
# This only returns documents from the MLOps Zoomcamp.
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [20]:
# let us wrap the search functionality into a function so we can reuse it easily.
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
search_results = search(question)
print(f"question: {question}")
search_results

question: I just discovered the course. Can I join now?


[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

### Building the Prompt

In [23]:
# The instructions tell the LLM its role and how to answer:
INSTRUCTIONS = """
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. 
If the answer is not found in the context, respond with "I don't know."
"""

In [24]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [25]:
# The context is a formatted string with all the search results:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [26]:
print(build_context(search_results))

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [27]:
# Now we combine the question with the context into the user prompt
# You should see a prompt with the question at the top and several FAQ entries below it.
# This is exactly what we'll send to the LLM.
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [28]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

### The LLM

In [32]:
# The last component of our RAG pipeline is the LLM. 
# It takes the prompt we built and generates an answer.
# The response is a Pydantic object. 
# The answer is in response.output - a list of output items.
response = llm(user_prompt=prompt)
print(response)

Yes, you can join now and start learning!

The videos and GitHub materials are available for you to start whenever you want.

However, if you want to receive a certificate, you need to:
1.  Submit your project while submissions are still being accepted.
2.  Complete the course with a "live" cohort, which includes submitting your project and peer-reviewing other projects during the designated period. Certificates are not awarded for self-paced mode outside of these deadlines.


In [34]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

# Combine messages for the API call, preserving role information in the content
combined_content = ""
for msg in message_history:
    combined_content += f"[{msg['role'].upper()}]\n{msg['content']}\n\n"

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=combined_content,
)

print(response.text)

Yes, you can join now. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.


### Full RAG

In [35]:
# With search, the prompt, and the LLM ready, we wire them together:
def rag(query, model="gemini-2.5-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [36]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start whenever you want, as the videos and GitHub materials are available.


In [37]:
answer = rag("How do I get a certificate?")
print(answer)

To get a certificate, you need to:
1.  Finish the course with a "live" cohort.
2.  Pass the Capstone project. This involves submitting your project and then peer-reviewing 3 capstones.
3.  Peer-reviewing can only be done at the time the course is running.

You will not receive a certificate if you follow the course in a self-paced mode. Missing homework does not prevent you from getting a certificate, as long as you pass the Capstone project.
